# LIBERO v2 — quickstart access guide

Practical walkthrough of how to read trials from the v2 HDF5 dataset:

1. Open the dataset (two equivalent paths: high-level `LiberoV2Dataset` vs raw `h5py`).
2. The full list of per-trial fields, what they mean, and how to access each.
3. Group contacts across trials — by failure mode, by task, by `is_holding`, by `(demo, bin)` sibling group.
4. Split a contact pool into train/val/test by trial keys.

Companion: `explore_libero_v2.ipynb` (visual inspection, storyboards, throughput timing). This notebook is *all* about access patterns, no plotting beyond a single sanity check at the end.

In [ ]:
# REQUIRED setup. Three things must happen BEFORE h5py and planner imports:
#  - sys.path / chdir so 'planner' resolves (Jupyter's CWD is notebooks/).
#  - HDF5_USE_FILE_LOCKING=FALSE: dataset is one-writer-per-file by design,
#    locks are unnecessary and break cross-process reads on ext4.
#  - import hdf5plugin: registers the blosc filter the writer used so
#    `f[k][()]` decompresses successfully (without it you get a confusing
#    'plugin directory not found' OSError).
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.getcwd() != REPO_ROOT:
    os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'
import hdf5plugin  # noqa: F401  — must come before/at h5py import time

from pathlib import Path
import h5py
import numpy as np
import pandas as pd

V2_ROOT = Path('/media/aaron/F/failbench/libero/v2')
SPLITS = ('libero_spatial', 'libero_object', 'libero_goal')
print('cwd:', os.getcwd())
print('v2 root:', V2_ROOT, '— exists:', V2_ROOT.exists())

## 1. Two ways to open the dataset

**Option A — `LiberoV2Dataset` (high level).** PyTorch-style dataset. Use this when you want a single `__getitem__` that returns a sample dict with sensible defaults (window + both cams + RGB + depth + goal + contacts). Configurable feature toggles to drop streams cheaply.

**Option B — raw `h5py`.** Use this when you want field-level control, partial reads, or to iterate over many trials inside a single per-task file (fastest path for analysis loops). The v2 store guarantees one `<task>.h5` per task, with all trials under `/trials/<trial_id>/`.

In [ ]:
# Option A: high-level dataset class
from planner.risk.dataset_v2 import LiberoV2Dataset

ds = LiberoV2Dataset(
    V2_ROOT,
    splits=SPLITS,
    use_window=True,       # T=8 window arrays
    use_wrist_cam=True,    # both cameras
    use_depth=True,        # RGB+depth
    use_settle=False,      # set True if you need the S=50 settle trajectory
    use_failure_mode=True, # adds a 5-D failure_mode_onehot field
)
print(f'Total trials: {len(ds):,}')
sample = ds[0]
print(f'Sample dict has {len(sample)} keys.')

In [ ]:
# Option B: raw HDF5
task_path = V2_ROOT / 'libero_spatial' / 'pick_up_the_black_bowl_on_the_stove_and_place_it_on_the_plate.h5'
with h5py.File(task_path, 'r') as f:
    print('file attrs:', dict(f.attrs.items()))
    trial_ids = sorted(f['trials'].keys())
    print(f'trials: {len(trial_ids)} (first 3 = {trial_ids[:3]})')
    g = f[f'trials/{trial_ids[0]}']
    print(f'\nDatasets in this trial: {len(list(g.keys()))}')
    print(f'Attrs on this trial:    {len(list(g.attrs.keys()))}')

## 2. Field-by-field access reference

All shapes assume `T=8` (window), `K=3` (goal lookahead), `S=50` (settle samples), image size `240×320`, and `n_obj` varying per scene (commonly 8–12 for LIBERO).

**With the high-level dataset** (`s = ds[i]`), every field below is a numpy array under the corresponding key, except scalar attrs which are returned as Python `int`/`float`/`str`/`bool`.

**With raw HDF5** (`g = f['trials/<trial_id>']`):
- *Datasets* are accessed as `g['name'][()]` (load whole array) or with slicing like `g['name'][:3]` (lazy partial read).
- *Attrs* are accessed as `g.attrs['name']` and decoded as below.

Strings in attrs may come back as `bytes` (HDF5 fixed-length convention); use `s.decode('utf-8') if isinstance(s, bytes) else s`.

### 2a. Pre-failure window (NEW in v2)

T=8 frames ending at `fail_idx`. Use these for any model that benefits from motion history.

In [ ]:
with h5py.File(task_path, 'r') as f:
    g = f[f'trials/{trial_ids[0]}']

    # Image stacks. Indexing g['name'][k] reads only the k-th frame (chunked I/O).
    window_av_rgb   = g['window_agentview_rgb'][()]        # (T, 240, 320, 3) uint8
    window_av_depth = g['window_agentview_depth'][()]      # (T, 240, 320)    float16 (metres)
    window_wr_rgb   = g['window_wrist_rgb'][()]            # (T, 240, 320, 3) uint8
    window_wr_depth = g['window_wrist_depth'][()]          # (T, 240, 320)    float16

    # Per-frame state. window_qvel is REAL finite-diff'd velocity (vs v1's zero).
    window_frame_idx    = g['window_frame_idx'][()]        # (T,)    i32 — demo timesteps
    window_qpos         = g['window_qpos'][()]             # (T, 7)  f32
    window_qvel         = g['window_qvel'][()]             # (T, 7)  f32
    window_ee_pos       = g['window_ee_pos'][()]           # (T, 3)  f32
    window_gripper_ctrl = g['window_gripper_ctrl'][()]     # (T, 1)  f32 — mean finger qpos

print('window_qpos[-1] (state at fail moment):', window_qpos[-1])
print('window_qvel[-1] (real velocity at fail):', window_qvel[-1])
print(f'window_av_rgb shape={window_av_rgb.shape} dtype={window_av_rgb.dtype}')

### 2b. Goal / intent feature

K=3 future commanded states from the *un-failed* demo at offsets `+5/+15/+30` demo steps after `fail_idx`. Tells the model what the controller *was about to do*.

In [ ]:
with h5py.File(task_path, 'r') as f:
    g = f[f'trials/{trial_ids[0]}']
    goal_qpos    = g['goal_qpos'][()]            # (K, 7)
    goal_qvel    = g['goal_qvel'][()]            # (K, 7)
    goal_ee_pos  = g['goal_ee_pos'][()]          # (K, 3)
    goal_grip    = g['goal_gripper_ctrl'][()]    # (K, 1)
    goal_offsets = g['goal_offsets'][()]         # (K,) e.g. [5, 15, 30]

print('goal_offsets:', goal_offsets)
print('goal_ee_pos  :', goal_ee_pos)
print('goal_gripper at each offset:', goal_grip.flatten())

### 2c. Single-frame fields at `fail_idx` (v1-compat)

Strict superset of v1's npz schema. `pre_qpos` is identical to `window_qpos[-1]` by construction.

In [ ]:
with h5py.File(task_path, 'r') as f:
    g = f[f'trials/{trial_ids[0]}']
    pre_qpos          = g['pre_qpos'][()]                # (7,)   f64
    pre_qvel          = g['pre_qvel'][()]                # (7,)
    pre_ee_pos        = g['pre_ee_pos'][()]              # (3,)
    pre_gripper_ctrl  = g['pre_gripper_ctrl'][()]        # (1,)
    pre_target_qpos   = g['pre_target_qpos'][()]         # (7,)  — controller hold target during settle
    pre_rgb           = g['pre_rgb'][()]                 # (240, 320, 3) uint8
    pre_depth         = g['pre_depth'][()]               # (240, 320) f16
    pre_wrist_rgb     = g['robot0_eye_in_hand_rgb'][()]  # (240, 320, 3) uint8
    pre_wrist_depth   = g['robot0_eye_in_hand_depth'][()]

print('pre_qpos == window_qpos[-1]?', np.allclose(pre_qpos, window_qpos[-1]))

### 2d. Contacts (the label)

Variable-length `N` per trial. World-frame positions are the geometric truth — project them anywhere later.

In [ ]:
with h5py.File(task_path, 'r') as f:
    # Pick a trial with a sizeable contact cloud for illustration.
    for tid in trial_ids:
        if f[f'trials/{tid}']['contact_positions'].shape[0] > 500:
            break
    g = f[f'trials/{tid}']
    print(f'illustrating with {tid}, mode={g.attrs["failure_mode"]}')

    contact_positions   = g['contact_positions'][()]     # (N, 3) f32   world frame, accumulated over 500 settle steps
    contact_forces      = g['contact_forces'][()]        # (N, 6) f32   MuJoCo contact-frame (3 linear + 3 torque)
    contact_force_world = g['contact_force_world'][()]   # (N, 3) f32   linear force rotated to world frame
    contact_time        = g['contact_time'][()]          # (N,)   i32   settle-step index in [0, 500)
    contact_geom_pairs  = g['contact_geom_pairs'][()]    # (N, 2) i32   MuJoCo geom IDs
    impacted_geom_ids   = g['impacted_geom_ids'][()]     # (M,)   i32   sorted union of touched geoms

    print(f'N contacts: {len(contact_positions)}')
    print(f'Time range: [{contact_time.min()}, {contact_time.max()}] of 500')
    print(f'Force-norm preservation check (rotation should preserve magnitude):')
    print(f'  contact-frame norms (first 5): {np.linalg.norm(contact_forces[:5, :3], axis=1)}')
    print(f'  world-frame   norms (first 5): {np.linalg.norm(contact_force_world[:5],  axis=1)}')

### 2e. Post-failure observations

Symmetric with pre — both cameras, RGB+depth, after the 500-step settle.

In [ ]:
with h5py.File(task_path, 'r') as f:
    g = f[f'trials/{trial_ids[0]}']
    post_av_rgb   = g['post_agentview_rgb'][()]
    post_av_depth = g['post_agentview_depth'][()]
    post_wr_rgb   = g['post_wrist_rgb'][()]
    post_wr_depth = g['post_wrist_depth'][()]
print(post_av_rgb.shape, post_av_depth.shape, post_wr_rgb.shape, post_wr_depth.shape)

### 2f. Camera calibration

Sufficient to project world-frame contacts into any camera, or unproject `(u, v, depth)` back to world. Agentview is static during the settle; the wrist cam moves with the arm, so its pose is stored per-window-frame.

In [ ]:
with h5py.File(task_path, 'r') as f:
    g = f[f'trials/{trial_ids[0]}']
    cam_av_pos   = g['cam_agentview_pos'][()]            # (3,)   f64
    cam_av_mat0  = g['cam_agentview_mat0'][()]           # (3, 3) f64 — RAW MuJoCo; apply Y-flip for image convention
    cam_av_fovy  = float(g['cam_agentview_fovy'][()])    # scalar — degrees
    cam_av_size  = g['cam_agentview_size'][()]           # (2,)   (W, H)

    cam_wr_pos_win  = g['cam_wrist_pos_window'][()]      # (T, 3)
    cam_wr_mat0_win = g['cam_wrist_mat0_window'][()]     # (T, 3, 3)
    cam_wr_fovy     = float(g['cam_wrist_fovy'][()])
    cam_wr_size     = g['cam_wrist_size'][()]

print('agentview FOV:', cam_av_fovy, 'deg   pos:', cam_av_pos)
print('wrist FOV:    ', cam_wr_fovy, 'deg')
print('wrist pos at each window frame (last 2):', cam_wr_pos_win[-2:])

### 2g. Object poses

Non-robot, non-table scene bodies, ordered by MuJoCo body id (stable within a demo MJCF, **not** shared across tasks). Always read `obj_names` alongside the pose arrays.

In [ ]:
with h5py.File(task_path, 'r') as f:
    g = f[f'trials/{trial_ids[0]}']
    obj_names    = [s.decode() if isinstance(s, bytes) else s for s in g['obj_names'][()]]
    obj_pos_pre  = g['obj_pos_pre'][()]           # (n_obj, 3) f32
    obj_quat_pre = g['obj_quat_pre'][()]          # (n_obj, 4) f32  (w, x, y, z)
    obj_pos_post = g['obj_pos_post'][()]
    obj_quat_post = g['obj_quat_post'][()]

print(f'{len(obj_names)} objects in this scene:')
for name, p_pre, p_post in zip(obj_names, obj_pos_pre, obj_pos_post):
    delta = np.linalg.norm(p_post - p_pre)
    print(f'  {name:50s}  pre z={p_pre[2]:.3f}  post z={p_post[2]:.3f}  Δ={delta:.4f} m')

### 2h. Settle state trajectory (for future world model)

Dense per-N-step snapshots (S=50 over 500 settle steps = every 10 steps ≈ 50 Hz). Robot + objects. Skip these if you don't need temporal cascade info — `LiberoV2Dataset(..., use_settle=False)` doesn't load them.

In [ ]:
with h5py.File(task_path, 'r') as f:
    g = f[f'trials/{trial_ids[0]}']
    settle_step_idx    = g['settle_step_idx'][()]         # (S,) i32
    settle_qpos        = g['settle_qpos'][()]             # (S, 7) f32
    settle_qvel        = g['settle_qvel'][()]             # (S, 7) f32
    settle_gripper_qpos= g['settle_gripper_qpos'][()]     # (S, 2) f32
    settle_obj_pos     = g['settle_obj_pos'][()]          # (S, n_obj, 3) f32
    settle_obj_quat    = g['settle_obj_quat'][()]         # (S, n_obj, 4) f32

print('settle steps sampled:', settle_step_idx)
print('arm qpos travel from start to end of settle (per-joint):')
print((settle_qpos[-1] - settle_qpos[0]))

### 2i. Failure descriptor + identity attrs

All scalar attrs live on the trial group (`g.attrs`). Decode bytes-strings if needed.

In [ ]:
with h5py.File(task_path, 'r') as f:
    g = f[f'trials/{trial_ids[0]}']

    def decode(x):
        return x.decode('utf-8') if isinstance(x, bytes) else x

    print(f"trial_id      : {decode(g.attrs['trial_id'])}")
    print(f"split / task  : {decode(g.attrs['split'])}  /  {decode(g.attrs['task'])}")
    print(f"demo_key      : {decode(g.attrs['demo_key'])}")
    print(f"seed / idx / bin : {g.attrs['seed']} / {g.attrs['seed_idx']} / {g.attrs['bin_idx']}")
    print(f"fail_idx      : {g.attrs['fail_idx']}")
    print(f"traj_progress : {g.attrs['traj_progress']:.3f}")
    print(f"failure_mode  : {decode(g.attrs['failure_mode'])}")
    print(f"failure_prob  : {g.attrs['failure_prob']:.3f}  (sampling prior, useful for re-weighting)")
    print(f"is_holding    : {bool(g.attrs['is_holding'])}")
    print(f"failure_joints (dataset): {g['failure_joints'][()]}  (1-based joint indices; empty for non-joint modes)")

    # Scene geometry attrs — used for world-frame label construction.
    print(f"\nscene_table_z : {g.attrs['scene_table_z']:.3f} m")
    print(f"scene_aabb_min: {g.attrs['scene_aabb_min']}")
    print(f"scene_aabb_max: {g.attrs['scene_aabb_max']}")
    print(f"robot_geom_ids (count): {len(g.attrs['robot_geom_ids'])}")

    # scene_entities_json is a json-serialised list of per-entity AABBs.
    import json
    entities = json.loads(decode(g.attrs['scene_entities_json']))
    print(f"\n{len(entities)} entities (showing first 3):")
    for e in entities[:3]:
        print(f"  {e['name']}: aabb_min={e['aabb_min']}, aabb_max={e['aabb_max']}")

## 3. Grouping all failure contacts across many trials

Three common analysis patterns. All use the manifest CSV first to filter at row level (fast, no HDF5 opens), then read just the needed trials.

### 3a. Load a manifest as a DataFrame

One manifest per split. Columns: `trial_id, split, task, demo_key, seed, seed_idx, bin_idx, fail_idx, traj_progress, failure_mode, failure_joints, failure_prob, is_holding, n_contacts, h5_path`.

In [ ]:
dfs = [pd.read_csv(V2_ROOT / s / 'manifest.csv') for s in SPLITS]
manifest = pd.concat(dfs, ignore_index=True)
print(f'Total rows: {len(manifest):,}')
manifest.head()

In [ ]:
# Quick distributions at the manifest level (no HDF5 opens).
print('Failure-mode distribution:')
print(manifest['failure_mode'].value_counts())
print('\nis_holding × failure_mode counts:')
print(pd.crosstab(manifest['failure_mode'], manifest['is_holding']))
print('\nContact-count stats by mode:')
print(manifest.groupby('failure_mode')['n_contacts'].agg(['median', 'mean', 'max', 'count']))

### 3b. Helper: stream contacts for a manifest-filtered set of trials

The pattern below opens each per-task HDF5 once, iterates the trials of interest inside it, and concatenates contact arrays. Keeps file-open overhead near-zero (the dominant cost) even when you ask for tens of thousands of trials.

In [ ]:
def gather_contacts(manifest_subset: pd.DataFrame,
                    fields=('contact_positions', 'contact_force_world', 'contact_time')
                    ) -> dict:
    """Concatenate contact arrays across the given manifest rows.

    Returns a dict like:
      {
        'contact_positions':   (sum_N, 3)  f32,
        'contact_force_world': (sum_N, 3)  f32,
        'contact_time':        (sum_N,)    i32,
        'trial_idx':           (sum_N,)    i32   <- index into manifest_subset for each contact
      }
    """
    out = {k: [] for k in fields}
    trial_idx_buf: list = []
    # Group by h5 path so we open each file once.
    for h5_path, rows in manifest_subset.groupby('h5_path', sort=False):
        with h5py.File(h5_path, 'r') as f:
            for local_idx, (manifest_i, row) in enumerate(rows.iterrows()):
                tid = row['trial_id']
                g = f[f'trials/{tid}']
                n = g['contact_positions'].shape[0]
                for k in fields:
                    out[k].append(g[k][()])
                trial_idx_buf.append(np.full(n, manifest_i, dtype=np.int32))
    out = {k: (np.concatenate(v) if v else np.empty((0,))) for k, v in out.items()}
    out['trial_idx'] = np.concatenate(trial_idx_buf) if trial_idx_buf else np.empty((0,), dtype=np.int32)
    return out

### 3c. Example: pool all SINGLE_JOINT contacts in libero_spatial

Filter the manifest, then call the helper. Returns one big concatenated array per field; `trial_idx` tells you which row of the manifest each contact came from (useful for back-joining to `failure_joints`, `task`, etc.).

In [ ]:
subset = manifest[(manifest['split'] == 'libero_spatial') &
                  (manifest['failure_mode'] == 'SINGLE_JOINT')]
print(f'Selected {len(subset):,} trials')

# CAUTION: this could be a lot of contacts (millions). For dev/exploration
# downsample first; for full analysis it still fits comfortably in 32 GB RAM.
pool = gather_contacts(subset.head(200))   # 200 trials is plenty for a sanity demo
print(f"  contact_positions   shape: {pool['contact_positions'].shape}")
print(f"  contact_force_world shape: {pool['contact_force_world'].shape}")
print(f"  unique trials in pool:     {len(np.unique(pool['trial_idx']))}")
print(f"  contact_time histogram (10 buckets over [0, 500]):")
hist, edges = np.histogram(pool['contact_time'], bins=10, range=(0, 500))
for h, e0, e1 in zip(hist, edges[:-1], edges[1:]):
    print(f"    [{int(e0):3d}, {int(e1):3d}): {h}")

### 3d. Splitting the pool by sub-condition

Common splits: by `failure_joints` (which joint failed), by `is_holding`, by `task`, or by `traj_progress` (early/mid/late failure). Always do these as manifest filters first, then `gather_contacts` per slice.

In [ ]:
# Per-joint pools (SINGLE_JOINT only)
for j_label in ('joint2', 'joint4', 'joint6'):
    slice_ = manifest[
        (manifest['failure_mode'] == 'SINGLE_JOINT') &
        (manifest['failure_joints'].fillna('') == j_label)
    ]
    print(f'  {j_label:8s}: {len(slice_):5d} trials, '
          f'mean n_contacts={slice_["n_contacts"].mean():6.1f}')

In [ ]:
# Holding vs not-holding × gripper-class failure
for mode in ('GRIPPER_OPEN', 'SLIPPERY_GRIP'):
    for h in (True, False):
        slice_ = manifest[(manifest['failure_mode'] == mode) &
                          (manifest['is_holding'] == h)]
        zero = (slice_['n_contacts'] == 0).mean()
        print(f'  {mode:14s} h={int(h)}: {len(slice_):5d} trials, '
              f'zero-contact rate {100*zero:5.1f}%')

In [ ]:
# Early / mid / late failure by traj_progress (use pd.cut for stable bins)
manifest['phase'] = pd.cut(manifest['traj_progress'], bins=[0, 0.33, 0.66, 1.0],
                           labels=['early', 'mid', 'late'])
print(manifest.groupby('phase', observed=True)['n_contacts'].agg(['mean', 'median', 'count']))

### 3e. Sibling-trial groups (same `(demo, bin)`, different sampled failure)

v1/v2 share the `(demo, seed_idx, bin_idx)` keying. For most demos there's 1 trial per `(demo, bin_idx)` — the 3 seeds×10 bins were used to **diversify** sampling, not to repeat the same (demo, bin).

In [ ]:
sibling_counts = manifest.groupby(['split', 'task', 'demo_key', 'bin_idx']).size().reset_index(name='n')
print('Distribution of sibling-group sizes:')
print(sibling_counts['n'].value_counts().sort_index())
# If we want all trials in a specific (demo, bin):
ex = sibling_counts[sibling_counts['n'] >= 2].iloc[0]
siblings = manifest[(manifest['split'] == ex['split']) &
                    (manifest['task'] == ex['task']) &
                    (manifest['demo_key'] == ex['demo_key']) &
                    (manifest['bin_idx'] == ex['bin_idx'])]
print(f"\nSibling group {ex['demo_key']}_b{ex['bin_idx']} ({ex['task'][:50]}...) has "
      f"{len(siblings)} sampled failures:")
print(siblings[['trial_id', 'failure_mode', 'failure_joints', 'n_contacts']])

## 4. Train / val / test splits

**Important: split by demo, not by trial.** If two trials share the same `demo_key`, their pre-failure scenes are nearly identical and leak across the train/val boundary. The keying that's safe to randomise on is `(split, task, demo_key)`.

If you want to also keep failure-mode balance, stratify on `failure_mode`.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

# Group key = the smallest unit that can't appear in both halves.
manifest['group_key'] = (manifest['split'] + '|'
                          + manifest['task'] + '|'
                          + manifest['demo_key'])

gss = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=0)
trainval_idx, test_idx = next(gss.split(manifest, groups=manifest['group_key']))

test = manifest.iloc[test_idx]
trainval = manifest.iloc[trainval_idx]

# Optionally split trainval into train+val with a second GSS on the same key.
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1 / 0.9, random_state=1)
train_idx, val_idx = next(gss2.split(trainval, groups=trainval['group_key']))
train = trainval.iloc[train_idx]
val   = trainval.iloc[val_idx]

print(f'train: {len(train):>6,} trials  ({train["group_key"].nunique():4d} demos)')
print(f'val  : {len(val):>6,} trials  ({val["group_key"].nunique():4d} demos)')
print(f'test : {len(test):>6,} trials  ({test["group_key"].nunique():4d} demos)')
leak = set(train['group_key']) & set(val['group_key']) | set(train['group_key']) & set(test['group_key'])
print(f'\nDemo-level overlap (should be 0): {len(leak)}')

In [ ]:
# Sanity: failure-mode balance preserved across splits?
for name, sub in [('train', train), ('val', val), ('test', test)]:
    print(f'{name:5s}:', dict(sub['failure_mode'].value_counts(normalize=True).round(3)))

## 5. Sanity-check projection: contacts overlaid on `post_agentview_rgb`

Quick visual to confirm the calibration is correct for downstream image-plane labelling. Uses the same Y-flip negation the v1 projector did.

In [ ]:
import matplotlib.pyplot as plt

# Pick a trial with plenty of contacts for a clear overlay.
row = manifest[manifest['n_contacts'].between(500, 5000)].iloc[0]
with h5py.File(row['h5_path'], 'r') as f:
    g = f[f"trials/{row['trial_id']}"]
    pts       = g['contact_positions'][()]
    post_rgb  = g['post_agentview_rgb'][()]
    cam_pos   = g['cam_agentview_pos'][()]
    cam_mat0  = g['cam_agentview_mat0'][()]
    fovy_deg  = float(g['cam_agentview_fovy'][()])
    W, H      = g['cam_agentview_size'][()]

# Pinhole projection with the v1-style Y-flip.
#   cam_mat0 is MuJoCo's camera->world rotation (cols = camera axes in world).
#   For pixel projection we want rows = camera axes -> take .T.
#   Then negate Y row so Y points DOWN in image convention.
#   MuJoCo cameras look along -Z, so depth in front of camera = -pc[:, 2].
fy = (H / 2.0) / np.tan(np.deg2rad(fovy_deg) / 2.0)
fx = fy
R  = cam_mat0.T.copy()
R[1] = -R[1]
pc    = (pts - cam_pos) @ R.T
depth = -pc[:, 2]                # positive depth = in front of camera
valid = depth > 1e-3
u = fx * pc[valid, 0] / depth[valid] + W / 2.0
v = fy * pc[valid, 1] / depth[valid] + H / 2.0
in_img = (u >= 0) & (u < W) & (v >= 0) & (v < H)

fig, ax = plt.subplots(figsize=(7, 5))
ax.imshow(post_rgb)
ax.scatter(u[in_img], v[in_img], s=3, alpha=0.4, c='red')
ax.set_title(f"{row['trial_id']}  \u00b7  {row['failure_mode']}  \u00b7  {len(pts)} contacts")
ax.axis('off')
plt.tight_layout()
plt.show()

## Cheat sheet

| Task | One-liner |
|---|---|
| Open one trial high-level | `s = LiberoV2Dataset(V2_ROOT, ...)[i]` |
| Open one trial low-level | `h5py.File(<h5>, 'r')['trials/<trial_id>']` |
| Filter trials by mode | `manifest[manifest['failure_mode'] == 'SINGLE_JOINT']` |
| Filter by joint | `manifest['failure_joints'].fillna('') == 'joint4'` |
| Filter by holding | `manifest['is_holding'] == True` |
| Filter by phase | `pd.cut(manifest['traj_progress'], bins=[0, .33, .66, 1])` |
| Pool contacts | `gather_contacts(manifest_subset)` (3b) |
| Demo-safe split | `GroupShuffleSplit(...).split(manifest, groups=manifest['group_key'])` |

Always remember:

- `os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'` before `import h5py`.
- `import hdf5plugin` at module level (registers blosc).
- Decode bytes attrs: `s.decode() if isinstance(s, bytes) else s`.
- `cam_*_mat0` is **raw MuJoCo** — apply `R[1] = -R[1]` for image-pixel convention.

## 6. Replay a full trial visually

A v2 trial spans three phases — **pre-failure window** (T=8 frames leading up to `fail_idx`), the **fail moment itself** (`pre_*` fields), and the **post-settle aftermath** (`post_*` fields). The 500-step physics settle in between is captured as state trajectory + accumulated contacts (no per-step pixels — re-renderable from `settle_*` later).

The cells below paint the full timeline.

First, pick a trial where the failure actually produced contacts so the visuals tell a story.

In [ ]:
# Choose a trial with a moderate-to-large contact cloud — gives us
# something interesting to see in both the cascade and the post frame.
trial_row = manifest[
    (manifest['n_contacts'].between(1000, 8000)) &
    (manifest['is_holding'] == True)
].iloc[2]
print(f"trial : {trial_row['trial_id']}")
print(f"task  : {trial_row['task']}")
print(f"split : {trial_row['split']}")
print(f"mode  : {trial_row['failure_mode']}   joints: {trial_row['failure_joints']}")
print(f"fail_idx={trial_row['fail_idx']}, traj_progress={trial_row['traj_progress']:.3f}")
print(f"is_holding={trial_row['is_holding']}, n_contacts={trial_row['n_contacts']}")

# Load the whole trial into memory once (≈ 8 MB uncompressed) for the rest of §6.
with h5py.File(trial_row['h5_path'], 'r') as f:
    g = f[f"trials/{trial_row['trial_id']}"]
    trial = {k: g[k][()] for k in g.keys()}
    trial_attrs = {k: (v.decode() if isinstance(v, bytes) else v)
                   for k, v in g.attrs.items()}
print('\nloaded keys:', sorted(trial.keys())[:8], '...')

### 6a. Pre-failure window storyboard — both cameras, RGB + depth

Four rows × T columns. Each column is one window frame, ordered left → right from `t = -7·stride` to `t = 0` (the fail moment). Reading top-to-bottom at any column gives you: agentview RGB, agentview depth, wrist RGB, wrist depth at that same instant.

Watch the agent arm move across the agentview row, and the gripper objects shift in the wrist row. Depth panels use a fixed range so brightness is comparable across frames.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

T = trial['window_agentview_rgb'].shape[0]
fig, axes = plt.subplots(4, T, figsize=(1.9 * T, 7))
row_labels = ['agentview\nRGB', 'agentview\ndepth (m)',
              'wrist\nRGB', 'wrist\ndepth (m)']
depth_vmin, depth_vmax = 0.3, 2.0

for k in range(T):
    axes[0, k].imshow(trial['window_agentview_rgb'][k])
    axes[0, k].set_title(f"frame {trial['window_frame_idx'][k]}\n(t-{T-1-k})", fontsize=9)
    axes[1, k].imshow(trial['window_agentview_depth'][k].astype(np.float32),
                      cmap='viridis', vmin=depth_vmin, vmax=depth_vmax)
    axes[2, k].imshow(trial['window_wrist_rgb'][k])
    axes[3, k].imshow(trial['window_wrist_depth'][k].astype(np.float32),
                      cmap='viridis', vmin=depth_vmin, vmax=depth_vmax)
    for r in range(4):
        axes[r, k].axis('off')

for r, lbl in enumerate(row_labels):
    axes[r, 0].text(-0.30, 0.5, lbl, transform=axes[r, 0].transAxes,
                    rotation=0, va='center', ha='right', fontsize=10, weight='bold')

fig.suptitle(
    f"Pre-failure window — {trial_row['trial_id']} ({trial_row['failure_mode']})",
    fontsize=12, y=1.01,
)
plt.tight_layout()
plt.show()

### 6b. Fail moment vs after-settle (single-frame comparison)

`pre_*` is the state at `fail_idx` (= last window frame); `post_*` is rendered after the 500-step settle. The deltas in object positions and depth are where the failure's damage shows up.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
imgs = [
    ('PRE agentview RGB', trial['pre_rgb'], dict()),
    ('PRE agentview depth', trial['pre_depth'].astype(np.float32),
     dict(cmap='viridis', vmin=depth_vmin, vmax=depth_vmax)),
    ('PRE wrist RGB', trial['robot0_eye_in_hand_rgb'], dict()),
    ('PRE wrist depth', trial['robot0_eye_in_hand_depth'].astype(np.float32),
     dict(cmap='viridis', vmin=depth_vmin, vmax=depth_vmax)),
    ('POST agentview RGB', trial['post_agentview_rgb'], dict()),
    ('POST agentview depth', trial['post_agentview_depth'].astype(np.float32),
     dict(cmap='viridis', vmin=depth_vmin, vmax=depth_vmax)),
    ('POST wrist RGB', trial['post_wrist_rgb'], dict()),
    ('POST wrist depth', trial['post_wrist_depth'].astype(np.float32),
     dict(cmap='viridis', vmin=depth_vmin, vmax=depth_vmax)),
]
for ax, (title, im, kw) in zip(axes.flat, imgs):
    ax.imshow(im, **kw)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
fig.suptitle('Pre (fail moment) vs Post (after 500-step settle)', fontsize=12)
plt.tight_layout()
plt.show()

### 6c. Contact accumulation over the settle

Each contact carries a `contact_time` ∈ [0, 500) — the settle step it landed on. Binning by time lets us see **how the contact cloud grew during the cascade**. We project the cumulative contact set at four equal-width time bins onto `post_agentview_rgb`.

Reads as a 4-frame stop-motion: first quarter (red), through to all 500 steps (yellow).

In [ ]:
# Pinhole projector (same convention as §5; MuJoCo cam looks along -Z).
cam_pos  = trial['cam_agentview_pos']
cam_mat0 = trial['cam_agentview_mat0']
fovy_deg = float(trial['cam_agentview_fovy'])
W, H     = trial['cam_agentview_size']

fy = (H / 2.0) / np.tan(np.deg2rad(fovy_deg) / 2.0); fx = fy
R  = cam_mat0.T.copy(); R[1] = -R[1]

def project(world_pts):
    pc = (world_pts - cam_pos) @ R.T
    depth = -pc[:, 2]
    in_front = depth > 1e-3
    u = fx * pc[in_front, 0] / depth[in_front] + W / 2.0
    v = fy * pc[in_front, 1] / depth[in_front] + H / 2.0
    in_img = (u >= 0) & (u < W) & (v >= 0) & (v < H)
    return u[in_img], v[in_img]

n_bins = 4
edges = np.linspace(0, 500, n_bins + 1).astype(int)
fig, axes = plt.subplots(1, n_bins, figsize=(4.5 * n_bins, 3.6))
for i, ax in enumerate(axes):
    mask = trial['contact_time'] < edges[i + 1]   # cumulative
    pts = trial['contact_positions'][mask]
    u, v = project(pts)
    ax.imshow(trial['post_agentview_rgb'])
    if len(u):
        ax.scatter(u, v, s=2, alpha=0.35,
                   c=plt.cm.YlOrRd(0.4 + 0.6 * (i + 1) / n_bins))
    ax.set_title(f'contacts t ≤ {edges[i+1]}  ({mask.sum()} pts)', fontsize=10)
    ax.axis('off')
fig.suptitle('Contact cloud growth across the 500-step settle', fontsize=12)
plt.tight_layout()
plt.show()

### 6d. Robot + object trajectories during the settle

Pure state — no rendering. `settle_qpos` (S=50 samples) tells you how the arm fell; `settle_obj_pos` shows which objects moved and by how much. The vertical line marks the failure injection (settle step 0).

For SINGLE_JOINT failures the failed joint typically drops monotonically while healthy joints chase the resistance PD target. For GRIPPER_OPEN with `is_holding=True` you'll see a sharp z-drop on whichever object was in the gripper.

In [ ]:
fig, (ax_q, ax_o) = plt.subplots(1, 2, figsize=(13, 4))

# (a) Robot arm qpos over settle. Highlight failed joints (1-based).
S = trial['settle_qpos'].shape[0]
t = trial['settle_step_idx']
failed = set(trial['failure_joints'].tolist())
for j in range(7):
    style = dict(lw=2.0, alpha=0.95) if (j + 1) in failed else dict(lw=1.0, alpha=0.45)
    label = f'joint{j+1}' + (' [FAILED]' if (j + 1) in failed else '')
    ax_q.plot(t, trial['settle_qpos'][:, j], label=label, **style)
ax_q.set_xlabel('settle step (0 = fail injection)')
ax_q.set_ylabel('qpos (rad)')
ax_q.set_title('Arm joint trajectory during settle')
ax_q.legend(fontsize=7, ncol=2, loc='best')
ax_q.grid(alpha=0.3)

# (b) Per-object world-z over settle. Things that fall stand out clearly.
obj_names = [s.decode() if isinstance(s, bytes) else s for s in trial['obj_names']]
for i, name in enumerate(obj_names):
    z_traj = trial['settle_obj_pos'][:, i, 2]
    delta = z_traj.max() - z_traj.min()
    if delta < 0.01:                  # skip static objects to reduce noise
        continue
    short = name.replace('_main', '').replace('_base', '')
    ax_o.plot(t, z_traj, lw=1.5, label=f'{short} (Δz={delta:.3f} m)')
ax_o.set_xlabel('settle step')
ax_o.set_ylabel('object z (m)')
ax_o.set_title('Object z-trajectories during settle (movers only)')
ax_o.legend(fontsize=7, loc='best')
ax_o.grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 6e. One-liner replay function

Wrap the four steps above into one function so you can replay any trial by trial_id (or by manifest row). Returns a single figure summarising the whole scenario.

In [ ]:
def replay_trial(manifest_row, save_to=None):
    """Render the full pre→fail→post→settle storyboard for one trial.

    Layout (rows top to bottom):
      1. Pre-window agentview RGB  (T frames)
      2. Pre-window wrist RGB      (T frames)
      3. Pre / post side-by-side both cameras  (4 panels in a separate row)
      4. Contact accumulation overlay (4 time bins on post_rgb)
      5. Arm + object trajectories during settle
    """
    with h5py.File(manifest_row['h5_path'], 'r') as f:
        g = f[f"trials/{manifest_row['trial_id']}"]
        tr = {k: g[k][()] for k in g.keys()}

    T = tr['window_agentview_rgb'].shape[0]
    fig = plt.figure(figsize=(2.0 * T, 12))
    gs = fig.add_gridspec(4, T, height_ratios=[1, 1, 1, 1.3], hspace=0.25)

    # Row 1: agentview RGB strip
    for k in range(T):
        ax = fig.add_subplot(gs[0, k])
        ax.imshow(tr['window_agentview_rgb'][k])
        ax.set_title(f"t-{T-1-k}", fontsize=8)
        ax.axis('off')
    fig.text(0.005, 0.86, 'agentview RGB', rotation=90, va='center', fontsize=9, weight='bold')

    # Row 2: wrist RGB strip
    for k in range(T):
        ax = fig.add_subplot(gs[1, k])
        ax.imshow(tr['window_wrist_rgb'][k])
        ax.axis('off')
    fig.text(0.005, 0.62, 'wrist RGB', rotation=90, va='center', fontsize=9, weight='bold')

    # Row 3: pre/post side-by-side, 4 columns (use first 4 grid slots, fill rest empty)
    pre_post = [
        ('PRE agent',  tr['pre_rgb']),
        ('POST agent', tr['post_agentview_rgb']),
        ('PRE wrist',  tr['robot0_eye_in_hand_rgb']),
        ('POST wrist', tr['post_wrist_rgb']),
    ]
    for k, (lbl, im) in enumerate(pre_post):
        if k < T:
            ax = fig.add_subplot(gs[2, k])
            ax.imshow(im); ax.set_title(lbl, fontsize=9); ax.axis('off')

    # Row 4: contact accumulation across 4 time bins
    cam_pos = tr['cam_agentview_pos']; cam_mat0 = tr['cam_agentview_mat0']
    fovy = float(tr['cam_agentview_fovy']); Wp, Hp = tr['cam_agentview_size']
    fy = (Hp/2)/np.tan(np.deg2rad(fovy)/2); fx = fy
    R = cam_mat0.T.copy(); R[1] = -R[1]
    edges = np.linspace(0, 500, 5).astype(int)
    for k in range(min(4, T)):
        ax = fig.add_subplot(gs[3, k])
        ax.imshow(tr['post_agentview_rgb'])
        mask = tr['contact_time'] < edges[k + 1]
        pts = tr['contact_positions'][mask]
        if len(pts):
            pc = (pts - cam_pos) @ R.T
            d = -pc[:, 2]; ok = d > 1e-3
            u = fx * pc[ok, 0] / d[ok] + Wp/2
            v = fy * pc[ok, 1] / d[ok] + Hp/2
            inimg = (u >= 0) & (u < Wp) & (v >= 0) & (v < Hp)
            ax.scatter(u[inimg], v[inimg], s=1.5, alpha=0.35, c='red')
        ax.set_title(f"contacts t<{edges[k+1]} ({mask.sum()})", fontsize=8)
        ax.axis('off')

    fig.suptitle(
        f"{manifest_row['trial_id']} | {manifest_row['task'][:60]} | "
        f"{manifest_row['failure_mode']} joints={manifest_row['failure_joints']} "
        f"holding={manifest_row['is_holding']} N={manifest_row['n_contacts']}",
        fontsize=11, y=0.995,
    )
    if save_to:
        fig.savefig(save_to, dpi=120, bbox_inches='tight')
    return fig

# Replay the same trial we've been inspecting above.
_ = replay_trial(trial_row)
plt.show()

### 6f. Replay a few different failure flavours side by side

Pick one example per failure mode, all from the same split/task family, to compare how the cascade looks under different failure types.

In [ ]:
for mode in ['GRIPPER_OPEN', 'SINGLE_JOINT', 'MULTI_JOINT', 'ALL_JOINTS']:
    candidates = manifest[
        (manifest['failure_mode'] == mode) &
        (manifest['is_holding'] == True) &
        (manifest['n_contacts'].between(500, 10000)) &
        (manifest['split'] == 'libero_spatial')
    ]
    if not len(candidates):
        continue
    _ = replay_trial(candidates.iloc[0])
    plt.show()